**MICOM ANALYSIS**

The metabolic modeling pipeline provides a complete analysis (i.e., prediction and validation) of microbial communities and their functional activities using AD, MCI, Healthy sample groups. In detail, the notebook installs and initializes the environment, as well as MICOM and the AGORA2 database, and uses the relative abundance data to create the input for constructing the community model. We also create a community model based on genera and check for missing or faulty models. Additionally, we create model-specific MICOM communities, complete the flux balance analysis of the models for the three sample groups, and determine all of the fluxes in the models and the growth rates of all of the communities. We prepare all of the metabolic features for further comparisons to the machine learning analyses.

In [ ]:
# Purpose: This notebook performs genome-scale metabolic modeling of the gut microbiome using MICOM to generate functional features for downstream analysis.
# Description: It builds sample-specific microbial community models using AGORA2 genus-level GEMs, estimates community growth rates, and computes reaction fluxes that represent predicted metabolic activity across AD, MCI, and Healthy groups.


#Main Steps Included:

#Environment setup and installation of MICOM and AGORA2 metabolic model database

#Formatting and validating relative abundance tables for MICOM compatibility

#Building genus-level community metabolic models for each sample

#Checking for missing GEMs or unmodeled taxa

#Running flux balance analysis (FBA) to obtain reaction flux predictions

#Extracting community growth rates and per-reaction flux values

#Saving processed metabolic features for integration with RA features in ML models


#Key Tools and Libraries Used:

#MICOM: For constructing community metabolic models and running FBA

#AGORA2 GEMs: Genome-scale metabolic models used as metabolic references

#Pandas: Data loading, merging, and cleaning

#NumPy: Numerical operations and processing

#Pathlib / os: File handling and directory organization

#Matplotlib / Seaborn: Exploratory visualizations of metabolic outputs

Connect to Google Drive and Create MICOM Folder

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Creat main folder for project inputs and outputs
import os
micom_path = '/content/drive/MyDrive/MICOM'
os.makedirs(micom_path, exist_ok=True)

Install MICOM and AGORA Database

In [ ]:
# Install MICOM and dependencies
!pip install micom
!pip install cobra optlang swiglpk

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import shutil

import cobra
from micom.workflows import build, build_database
from micom import load_pickle

In [ ]:
# Install AGORA
# Use the shell command !wget to download AGORA2 models as a zip file
!wget "https://www.vmh.life/files/reconstructions/AGORA2/version2.01/sbml_files_fixed/zipped/AGORA2_models/AGORA2_SBML.zip"

In [ ]:
# Define the path for the AGORA Database
agora_path = Path(micom_path) / "AGORA2_models"

In [ ]:
# Extract the models
!unzip AGORA2_SBML.zip -d {agora_path}

In [ ]:
# Remove zip file
!rm AGORA2_SBML.zip

Build Class for MICOM Pre-processing and Analysis


In [ ]:
# Create a class to handle pre-processing and analysis functions.
class MICOM_Analysis:
#-------------------------------------------------------------------------------------------------------------------------------------------
  def __init__(self, rel_abd, condition):
    '''
    Constructor Method: Creates instance of the class with the following attributes:
          self - reference to the class object
          rel_abd - dataframe of relative abundance data
          condition - string of condition name
    '''
    self.rel_abd = rel_abd
    self.condition = condition

    # Creat a subfolder in MICOM folder named /condition_communities
    # Will be used to store .pickle files
    self.community_folder = Path(micom_path) / f"{self.condition}_communities"
    os.makedirs(self.community_folder, exist_ok=True)

    # Create an attribute to store the community manifest
    self.community_manifest = None
#-------------------------------------------------------------------------------------------------------------------------------------------
  def format_rel_abd(self):
    '''
    Pre-Processing Method: Formats the relative abundance dataframe
          self - reference to the class object
          Tasks:
          - Renames the "#OTU ID" column to "id" as required for MICOM analysis
          - Extracts genus names from the "id" column and creates a new "genus" column
          - Reshapes the dataframe to a long format (required by MICOM)
          - Changes "id" column to genus name (required by MICOM)
    '''
    # Rename the "#OTU ID" column to "id" as required for MICOM analysis
    self.rel_abd = self.rel_abd.rename(columns={"#OTU ID": "id"})

    # Create an empty list to store genus names
    genus_col = []
    for microbe in self.rel_abd["id"]:
      # For each microbe in the id column:
      # Split microbe into a list of ranks [domain, phylum, class, order, family, genus, species]
      lineage = microbe.split(';')
      # Initially set genus to "Unknown"
      genus = "Unknown"

      for rank in lineage:
        # For each rank in the lineage, check if it starts with "g__" for genus
        if rank.startswith("g__"):
          # If True, remove "g__" to leave only the genus name
          genus = rank.replace("g__", "")
          # Remove any brackets
          genus = genus.replace('[', '').replace(']', '')
          # If genus name is segmented by "-" or "_", rename to only the first part of the segment
          # Example: "Escherichia-Shigella" → "Escherichia"
          if '-' in genus:
            genus = genus.split('-')[0]
          if '_' in genus:
            genus = genus.split('_')[0]

      # Append genus name to genus_col, if genus was not classified it will remain as "Unknown"
      genus_col.append(genus)

    # Insert a column into the relative abundance dataframe, at index = 1 (second column), named
    # "genus", populated with values from genus_col
    self.rel_abd.insert(loc = 1, column = "genus", value = genus_col)

    # Reshape the dataframe to a long format (required by MICOM)
    # It keeps the "id" and "genus" columns as is
    # It creates a variable column that contains the names of the remaining columns (sample SRR IDs)
    # It creates a value column for the corresponding relative abundance values for each sample
    self.rel_abd = self.rel_abd.melt(
        id_vars = ["id", "genus"],   # remain unchanged
        var_name = "sample_id",      # name of variable column
        value_name = "abundance")    # name of value column

    # Change "id" column to genus name
    self.rel_abd["id"] = self.rel_abd["genus"]
#-------------------------------------------------------------------------------------------------------------------------------------------
  def genus_mapping(self, manifest):
    '''
    Maps each microbe in dataframe to a genus present in the AGORA database
          self - reference to the class object
          genus_database - list of genus names in the AGORA database
          Tasks:
          - Remove rows where the genus is "Unknown"
          - Remove rows where the genus cannot be mapped.

    # Remove sample where genus in Unknown
    self.rel_abd = self.rel_abd[self.rel_abd["genus"] != "Unknown"]
    # Check if sample genus exists in AGORA database, remove samples if not
    self.rel_abd = self.rel_abd[self.rel_abd["genus"].isin(genus_database)]
    # Reset dataframe index
    self.rel_abd = self.rel_abd.reset_index(drop=True)
    '''
    self.rel_abd = self.rel_abd[self.rel_abd["genus"] != "Unknown"].copy()

    # 2. Get allowed genera from manifest
    valid_genera = set(manifest["id"].unique())

    # 3. Keep only taxa that have a GEM
    self.rel_abd = self.rel_abd[self.rel_abd["genus"].isin(valid_genera)].copy()


    # 5. Reset index
    self.rel_abd = self.rel_abd.reset_index(drop=True)
#-------------------------------------------------------------------------------------------------------------------------------------------
  def build_communities(self, gem_folder):
    database_manifest_path = Path(gem_folder) / "manifest.csv"
    database_manifest = pd.read_csv(database_manifest_path)
    database_manifest = database_manifest[["genus", "file"]]

    taxonomy = self.rel_abd.copy()
    taxonomy = taxonomy.merge(
        database_manifest,
        on = "genus",
        how = "left")
    taxonomy = taxonomy[~taxonomy["file"].isna()].reset_index(drop=True)
    self.rel_abd = taxonomy

    self.community_manifest = build(
        taxonomy = self.rel_abd,
        model_db = None,
        out_folder = str(self.community_folder),
        cutoff = 0.0001,
        threads = 2)
#-------------------------------------------------------------------------------------------------------------------------------------------
  def check_communities(self):
    com_manifest_path = Path(self.community_folder) / "manifest.csv"

    if com_manifest_path.exists():
      self.community_manifest = pd.read_csv(com_manifest_path)

    else:
      self.community_manifest = []
      pickles = list(Path(self.community_folder).glob("*.pickle"))
      for community in pickles:
        sample_srr = community.stem
        self.community_manifest.append({
            "sample_id": sample_srr,
            "pickle_file": str(community)})

    self.community_manifest = pd.DataFrame(self.community_manifest)
    self.community_manifest.to_csv(com_manifest_path, index=False)

    self.rel_abd["sample_id"] = self.rel_abd["sample_id"].astype(str)
    valid_samples = set(self.community_manifest["sample_id"].astype(str))
    self.rel_abd = (self.rel_abd[self.rel_abd["sample_id"].isin(valid_samples)].reset_index(drop=True))
#-------------------------------------------------------------------------------------------------------------------------------------------


Format Relative Abundance TSVs

In [ ]:
# Read the relative abundance files into varaibles named after corresponding sample conditions
ad_rel_abd = pd.read_csv('/content/drive/MyDrive/Taxonomic_Identification/ad_species_relative_abundance.tsv', sep="\t")
mci_rel_abd = pd.read_csv('/content/drive/MyDrive/Taxonomic_Identification/mci_species_relative_abundance.tsv', sep="\t")
healthy_rel_abd = pd.read_csv('/content/drive/MyDrive/Taxonomic_Identification/healthy_species_relative_abundance.tsv', sep="\t")

In [ ]:
# Initialize MICOM_Analysis instances for AD, MCI, and Healthy datasets
ad = MICOM_Analysis(ad_rel_abd, "ad")
mci = MICOM_Analysis(mci_rel_abd, "mci")
healthy = MICOM_Analysis(healthy_rel_abd, "healthy")

In [ ]:
# Print the relative abundance datasets before pre-processing
print("ad", ad.rel_abd)
print("mci", mci.rel_abd)
print("healthy", healthy.rel_abd)

In [ ]:
# Format relative abundance tables to correct MICOM format, remove rows where genus is unknown
ad.format_rel_abd()
mci.format_rel_abd()
healthy.format_rel_abd()

In [ ]:
# Print the relative abundance datasets after pre-processing
print("ad", ad.rel_abd)
print("mci", mci.rel_abd)
print("healthy", healthy.rel_abd)

In [ ]:
# Save files for relative abundance datasets after pre-processing and before
# GEM filtration
ad.rel_abd.to_csv(Path(micom_path)/ "ad_rel_abd_frmt.csv", index=False)
mci.rel_abd.to_csv(Path(micom_path)/ "mci_rel_abd_frmt.csv", index=False)
healthy.rel_abd.to_csv(Path(micom_path)/ "healthy_rel_abd_frmt.csv", index=False)

Build Genera Database

In [ ]:
# List all the files in the AGORA database
agora_files = list(agora_path.glob("*.xml"))
# Initialize an empty list for the genera database
genera_database = []
# Create a folder for the GEM database
gem_folder = Path(micom_path) / "GEM_Database"
os.makedirs(gem_folder, exist_ok=True)

In [ ]:
def build_genera_database(ad, mci, healthy):
  '''
  Add all unique genera from AD, MCI, and Healthy datasets to the genera database
       ad - dataframe of relative abundance data from AD dataset
  	   mci - dataframe of relative abundance data from MCI dataset
  	   healthy - dataframe of relative abundance data from Healthy dataset
  '''
  for genus in ad:
    if pd.notna(genus) and genus != "Unknown" and genus not in genera_database:
      genera_database.append(genus)

  for genus in mci:
    if pd.notna(genus) and genus != "Unknown" and genus not in genera_database:
      genera_database.append(genus)

  for genus in healthy:
    if pd.notna(genus) and genus != "Unknown" and genus not in genera_database:
      genera_database.append(genus)

In [ ]:
# Call the build_genera_database using the "genus" columns from each dataframe
build_genera_database(ad.rel_abd["genus"], mci.rel_abd["genus"], healthy.rel_abd["genus"])

In [ ]:
# Print the genera database
print(genera_database)

Build GEM Database

In [ ]:
def build_gem_database(agora, database):
  '''
  Build a GEM database (manifest) consisting only of files found in the database
       agora - list of files
       database - list of genera in the database
  '''
  manifest = []

  for gem_model in agora:
    # For each GEM model file extract the microbe
    microbe = gem_model.stem
    # Split the microbe into ranks
    lineage = microbe.split("_")

    # Set genus to first rank in lineage
    genus = lineage[0]
    # Set species to second rank in lineage
    species = lineage[1]

    if genus in database:
      # If the genus is in the database, create a dictionary manifest with the above
      # information and append it to the manifest list
      manifest.append({
          "file": str(gem_model),
          "kingdom": "Bacteria",
          "phylum": "",
          "class": "",
          "order": "",
          "family": "",
          "genus": genus,
          "species": species})

  # Change manifest list to dataframe and return
  manifest = pd.DataFrame(manifest)
  return manifest

In [ ]:
# Build a manifest of AGORA GEM models that match the genera present in our samples
database_manifest = build_gem_database(agora_files, genera_database)

In [ ]:
# Use MICOM's build_database() method to generate .json files for each GEM in the filtered AGORA manifest
build_database(
    database_manifest,
    out_path = str(gem_folder),
    rank = "genus",
    threads = 2)

Process Missing GEMS

In [ ]:
def check_missing_gems():
  '''
  Check which required genera (present in sample datasets) are missing from the GEM database
  '''
  # List all files in the GEM database
  gem_files = list(gem_folder.glob("*.json"))
  # Create empty lists to store genera present and missing in the GEM database
  gem_database = []
  missing_gems = []

  # For each unique GEM in the GEM database, add it to gem_database list
  for gem in gem_files:
    gem = gem.stem
    if gem not in gem_database:
      gem_database.append(gem)

  # Check if each genus in the genera database (from sample datasets) has a corresponding
  # GEM model in the GEM database
  for genus in genera_database:
    # If not add to list of missing_gems
    if genus not in gem_database:
      missing_gems.append(genus)

    # If genus is present in both the GEM database and missing_gems list (if it was added later)
    # remove it from the missing_gems list
    elif genus in gem_database:
      if genus in missing_gems:
        missing_gems.remove(genus)

  # If there are no missing GEMS print a message that "All GEMs Found"
  if len(missing_gems) == 0:
    print("All GEMs Found")
  # Else print each of the genera missing GEM models
  else:
    for missing in missing_gems:
      print("GEM Not Found: ", missing)

  return missing_gems

In [ ]:
# Check for missing GEM models
missing_gems = check_missing_gems()

In [ ]:
def process_missing_gems(missing_database):
  '''
  Process AGORA database for missing and broken GEM files
        missing_database - list of genera missing from the GEM database
        Tasks:
        - Find whether missing GEM models have available or missing xml files
        - Check if available xml files are broken
        - Copy valid available xml files into a separate folder
        - Remove broken available xml files
        - Build GEM database manifest with only valid available xml files
  '''
  # Create empty list to store Path objects for available xml files
  available_xml = []
  # Create empty list to store genera with missing xml files
  missing_xml = []

  # For each missing GEM
  for gem in missing_database:
    found = False
    # Compare missing GEM to all xml files in AGORA database
    for xml in agora_files:
      xml_genus = xml.stem.split("_")[0]
      # If there are available files for the GEM add to available_xml list
      if gem == xml_genus:
        available_xml.append(xml)
        found = True
        break
    # If there are no matching xml files for the missing GEM add to missing_xml list
    if not found:
      missing_xml.append(gem)
#-------------------------------------------------------------------------------------------------------------------------------------------
  # Create empty lists to store Path objects for valid and broken available xml files
  valid_xml = []
  broken_xml = []

  # For each available xml, try to load it using COBRA
  for xml in available_xml:
    try:
      cobra.io.read_sbml_model(str(xml))
      # If it loads successfully, add it to list of valid_xml
      valid_xml.append(xml)
    # If an error is raised, print the error and Path of broken file
    # Add xml to list of broken_xml
    except Exception as error:
      broken_xml.append(xml)
      print(error, "\n")
      print("Broken SBML: ", xml, "\n")
#-------------------------------------------------------------------------------------------------------------------------------------------
  # Create folders to store valid and broken xml files
  valid_folder = Path(micom_path) / "Valid Missing xmls"
  broken_folder = Path(micom_path) / "Broken xmls"
  valid_folder.mkdir(exist_ok=True)
  broken_folder.mkdir(exist_ok=True)

  # Copy valid xmls into a separate folder
  for xml_file in valid_xml:
    valid_destination = Path(valid_folder) / xml_file.name
    shutil.copy(xml_file, valid_destination)

  # Move broken xml files into a separate folder
  for xml_file in broken_xml:
    broken_destination = Path(broken_folder) / xml_file.name
    xml_file.rename(broken_destination)

  # Build a GEM manifest from the valid xml GEM models and return it along with any genera still missing xml files
  manifest = build_gem_database(valid_xml, missing_database)
  return [manifest, missing_xml]

In [ ]:
missing_manifest = process_missing_gems(missing_gems)

In [ ]:
# Print list of GEM models missing xml files
print(missing_manifest[1])

In [ ]:
# Use MICOM's build_database() method to generate .json files for the missing GEMs with valid xml files
build_database(
    missing_manifest[0],
    out_path = str(gem_folder),
    rank = "genus",
    threads = 2)

In [ ]:
# Check for missing GEM models
missing_gems = check_missing_gems()

Update Database Manifest


In [ ]:
# List all GEMs in the final database
final_gems = list(gem_folder.glob("*.json"))

# Initialize an empty manifest
manifest = []
# For each GEM in the database, store its information in a dictionary and append to manifest
for gem in final_gems:
  genus = gem.stem
  manifest.append({
      "id": genus,
      "file": str(gem),
      "kingdom": "Bacteria",
      "phylum": "",
      "class": "",
      "order": "",
      "family": "",
      "genus": genus,
      "species": "",
      "summary_rank": "genus",
      "type": "json",
    })

# Creat database_manifest by converting manifest list into dataframe
database_manifest = pd.DataFrame(manifest)
# Ensure all file Paths are stored as strings
database_manifest["file"] = database_manifest["file"].astype(str)
# Save database_manifest as both a .json and .csv file in the GEM database folder
database_manifest.to_csv(gem_folder / "manifest.csv", index = False)

In [ ]:
# Map genera from dataframes to AGORA database, remove unmapped genera
ad.genus_mapping(database_manifest)
mci.genus_mapping(database_manifest)
healthy.genus_mapping(database_manifest)

In [ ]:
# Print final relative abundance tables
print("ad", ad.rel_abd)
print("mci", mci.rel_abd)
print("healthy", healthy.rel_abd)

Build Communities

In [ ]:
# Build MICOM community models for the AD samples
ad.build_communities(gem_folder)

In [ ]:
# Build community manifest
ad.check_communities()

In [ ]:
print(ad.rel_abd)

In [ ]:
# Build MICOM community models for the MCI samples
mci.build_communities(gem_folder)

In [ ]:
# Build community manifest
mci.check_communities()

In [ ]:
print(mci.rel_abd)

In [ ]:
# Build MICOM community models for the Healthy samples
healthy.build_communities(gem_folder)

In [ ]:
# Build community manifest
healthy.check_communities()

In [ ]:
print(healthy.rel_abd)

Identify Key Metabolites

In [ ]:
def run_cooperative_tradeoff(community, fraction = 0.7, fluxes = False, pfba = True):
  cooperative_tradeoff = community.cooperative_tradeoff(
      fraction = fraction,
      fluxes = fluxes,
      pfba = pfba)

  return cooperative_tradeoff

def test_community_flux(community_list):
  com_fluxes = []
  sample_srr = []

  for community in community_list:
    test_community = load_pickle(community)
    # Run FBA analysis to get metabolic fluxes
    test_solution = run_cooperative_tradeoff(test_community, fraction = 0.7, fluxes = True, pfba = True)
    # Save table of flux (rows = taxa/medium, cols = reactions)
    fluxes = test_solution.fluxes

    fluxes = fluxes.loc["medium"].dropna()
    sorted_flux = fluxes.reindex(fluxes.abs().sort_values(ascending=False).index)

    com_fluxes.append(sorted_flux)
    sample_srr.append(community.stem)
    print(len(sample_srr))

  com_fluxes = pd.DataFrame(com_fluxes, index=sample_srr)
  return com_fluxes

def reaction_counts(com_fluxes):
  counts = com_fluxes.notna().sum().sort_values(ascending=False)
  return counts

def flux_ratios(ad_fluxes, mci_fluxes, healthy_fluxes):
  # ratio of metabolite flux being present in a sample
  counts = pd.DataFrame({
      "AD_count": ad_fluxes.notna().sum(),
      "MCI_count": mci_fluxes.notna().sum(),
      "Healthy_count": healthy_fluxes.notna().sum()})

  ratios = pd.DataFrame({
      "AD_ratio": counts["AD_count"] / ad_fluxes.shape[0],
      "MCI_ratio": counts["MCI_count"] / mci_fluxes.shape[0],
      "Healthy_ratio": counts["Healthy_count"] / healthy_fluxes.shape[0]})

  # Find difference btw max and min ratio, if ratio for each groups is similar
  # range will be small, if ratio for each group is different, range will be large
  ratios["presence_range"] = ratios.max(axis = 1) - ratios.min(axis = 1)
  ratios = ratios.sort_values("presence_range", ascending=False)

  return ratios

def community_growth_table(community_list):
  """
  Compute community growth rate using cooperative tradeoff.
  Returns a DataFrame indexed by sample_id with one column:
      - community_growth
  """

  rows = []

  for sample in community_list:
    sample_srr = sample.stem
    print(sample_srr)

    # Load community
    community = load_pickle(sample)

    # Run cooperative tradeoff
    tradeoff_solution = run_cooperative_tradeoff(community, fraction = 0.7, fluxes = False, pfba = True)

    # Extract growth safely (handle different MICOM versions)
    if hasattr(tradeoff_solution, "community_growth"):
      growth = tradeoff_solution.community_growth
    elif hasattr(tradeoff_solution, "growth_rate"):
      growth = tradeoff_solution.growth_rate
    else:
      growth = None

    rows.append({
        "sample_id": sample_srr,
        "community_growth": float(growth) if growth is not None else None})

  growth_df = pd.DataFrame(rows).set_index("sample_id")
  return growth_df

In [ ]:
def run_micom_analysis(condition):
  community_list = list(Path(condition.community_folder).glob("*.pickle"))
  fluxes = test_community_flux(community_list)
  counts = reaction_counts(fluxes)
  growth = community_growth_table(community_list)

  return fluxes, counts, growth

In [ ]:
ad_micom_analysis = run_micom_analysis(ad)
ad_fluxes = ad_micom_analysis[0]
ad_counts = ad_micom_analysis[1]
ad_growth = ad_micom_analysis[2]

In [ ]:
print(ad_fluxes.head())

In [ ]:
print(ad_counts)

In [ ]:
print(ad_growth)

In [ ]:
ad_mean_growth = ad_growth["community_growth"].mean()
print(ad_mean_growth)

In [ ]:
mci_micom_analysis = run_micom_analysis(mci)
mci_fluxes = mci_micom_analysis[0]
mci_counts = mci_micom_analysis[1]
mci_growth = mci_micom_analysis[2]

In [ ]:
print(mci_fluxes.head())

In [ ]:
print(mci_counts)

In [ ]:
print(mci_growth)

In [ ]:
mci_mean_growth = mci_growth["community_growth"].mean()
print(mci_mean_growth)

In [ ]:
healthy_micom_analysis = run_micom_analysis(healthy)
healthy_fluxes = healthy_micom_analysis[0]
healthy_counts = healthy_micom_analysis[1]
healthy_growth = healthy_micom_analysis[2]

In [ ]:
print(healthy_fluxes.head())

In [ ]:
print(healthy_counts)

In [ ]:
print(healthy_growth)

In [ ]:
healthy_mean_growth = healthy_growth["community_growth"].mean()
print(healthy_mean_growth)

Feature Extraction

In [ ]:
# Note: negative flux means secretion, positive means uptake

In [ ]:
# Mean flux
ad_mean = ad_fluxes.mean(axis=0)
mci_mean = mci_fluxes.mean(axis=0)
healthy_mean = healthy_fluxes.mean(axis=0)

In [ ]:
flux_means = pd.DataFrame({
    "AD_mean": ad_mean,
    "MCI_mean": mci_mean,
    "Healthy_mean": healthy_mean})

In [ ]:
# add variance column
flux_means["variance"] = flux_means.max(axis=1) - flux_means.min(axis=1)

In [ ]:
# sort by variance
flux_means = flux_means.sort_values("variance", ascending=False)

In [ ]:
print(flux_means.head(10))

In [ ]:
# Shorten to top 5
top_flux = flux_means.head(5).index.tolist()
print(top_flux)

In [ ]:
ad_top_flux = ad_fluxes[top_flux].copy()
mci_top_flux = mci_fluxes[top_flux].copy()
healthy_top_flux = healthy_fluxes[top_flux].copy()

In [ ]:
# Add community growth column
ad_top_flux["growth"] = ad_growth["community_growth"]
mci_top_flux["growth"] = mci_growth["community_growth"]
healthy_top_flux["growth"] = healthy_growth["community_growth"]

In [ ]:
print("Top 5 Reactions in AD:")
print(ad_top_flux.head())
print("\nTop 5 Reactions in MCI:")
print(mci_top_flux.head())
print("\nTop 5 Reactions in Healthy:")
print(healthy_top_flux.head())

In [ ]:
# Add condition labels
ad_top_flux["condition"] = "AD"
mci_top_flux["condition"] = "MCI"
healthy_top_flux["condition"] = "Healthy"

In [ ]:
# Combine into one ML dataframe
ml_flux_df = pd.concat([ad_top_flux, mci_top_flux, healthy_top_flux])

In [ ]:
print(ml_flux_df.shape)
ml_flux_df.head()

In [ ]:
# Add sample id column
ml_flux_df["sample_id"] = ml_flux_df.index
cols = ["sample_id", "condition"] + [c for c in ml_flux_df.columns if c not in ["sample_id", "condition"]]
ml_flux_df = ml_flux_df[cols]
ml_flux_df = ml_flux_df.reset_index(drop=True)

In [ ]:
# Create folder for Machine Learning Analysis
ml_analysis_path = "/content/drive/MyDrive/Machine_Learning"
os.makedirs(ml_analysis_path, exist_ok=True)

In [ ]:
# Export MICOM Features to folder
save_path = Path(ml_analysis_path) / "micom_features.csv"
ml_flux_df.to_csv(save_path, index=False)